# Conversao ODBC (`base.csv`) para formato de fechamento

Este notebook converte os dados de `02-Referencias/Meus_Dados/base.csv` para o mesmo layout de colunas do arquivo `02-Referencias/FECHAMENTO GERAL 2026 (HUGO).csv`.

## Como usar
1. Ajuste `MES_REFERENCIA` e, se quiser, `CAMPO_DATA_FILTRO`.
2. Execute todas as celulas.
3. O arquivo convertido sera salvo em `02-Referencias/` no formato Excel (`.xlsx`).


In [1]:
from pathlib import Path
import re
import pandas as pd
import numpy as np

# Parametros principais
MES_REFERENCIA = '04/2026'            # formato MM/AAAA
CAMPO_DATA_FILTRO = 'lancamento'      # 'lancamento' (dd/mm/aaaa) ou 'ite_pagrec_vencimento' (aaaa-mm-dd)

# Caminhos
REFS_DIR = Path('..') / '..' / '02-Referencias'
BASE_PATH = REFS_DIR / 'Meus_Dados' / 'base.csv'
MODELO_PATH = REFS_DIR / 'FECHAMENTO GERAL 2026 (HUGO).csv'
OUT_DIR = REFS_DIR

if not BASE_PATH.exists():
    raise FileNotFoundError(f'Arquivo nao encontrado: {BASE_PATH.resolve()}')
if not MODELO_PATH.exists():
    raise FileNotFoundError(f'Arquivo nao encontrado: {MODELO_PATH.resolve()}')

print('Parametros carregados com sucesso.')
print(f'MES_REFERENCIA={MES_REFERENCIA} | CAMPO_DATA_FILTRO={CAMPO_DATA_FILTRO}')


Parametros carregados com sucesso.
MES_REFERENCIA=04/2026 | CAMPO_DATA_FILTRO=lancamento


In [2]:
def to_float_br(v):
    if pd.isna(v):
        return np.nan
    s = str(v).strip()
    if s == '':
        return np.nan
    s = s.replace('.', '').replace(',', '.')
    try:
        return float(s)
    except ValueError:
        return np.nan

def to_br_number(v):
    if pd.isna(v):
        return ''
    s = f'{float(v):.2f}'
    return s.replace('.', ',')

def to_br_currency(v):
    if pd.isna(v):
        return ''
    txt = f'{float(v):,.2f}'
    txt = txt.replace(',', 'X').replace('.', ',').replace('X', '.')
    return f'R$ {txt}'

def split_descen(descen):
    partes = [p.strip() for p in str(descen).split('/') if p.strip()]
    natureza = partes[0] if len(partes) > 0 else ''
    divisao = partes[1] if len(partes) > 1 else ''
    filial_cc = partes[2] if len(partes) > 2 else ''
    resto = partes[3:] if len(partes) > 3 else []
    n3 = resto[0] if len(resto) > 0 else filial_cc
    n4 = ' / '.join(resto) if len(resto) > 0 else n3
    # ODBC as vezes repete o nome do nivel 3 no inicio de n4 (ex.: "PRENSA MOVEL / QXH2G14 (PHH0061)")
    if n3 and isinstance(n4, str) and n4.startswith(n3 + ' / '):
        n4 = n4[len(n3) + 3:].strip()
    # Quando n3 perde sufixo (ex.: JOINVILLE/SC), n4 pode ficar "SC / EHH0044"
    if isinstance(n4, str) and ' / ' in n4:
        tail = n4.split(' / ')[-1].strip()
        if re.match(
            r'^([A-Z]{3}\d{4}|[A-Z]{3}\d[A-Z0-9]{3}|[A-Z0-9]{5,8}\s*\([^)]+\))$',
            tail,
            re.IGNORECASE,
        ):
            n4 = tail
    if isinstance(n4, str) and ' / ' in n4 and 'operadores' in n4.lower() and 'arcelor' in n4.lower():
        n4 = 'ARCELOR RESENDE - OPERADORES'
    return natureza, divisao, filial_cc, n3, n4

def levels_from_codcen(codcen):
    tokens = [t for t in str(codcen).strip().split('.') if t]
    if len(tokens) < 2:
        n1 = str(codcen).strip()
    else:
        n1 = '.'.join(tokens[:2])
    n2 = '.'.join(tokens[:3]) if len(tokens) >= 3 else n1
    n3 = '.'.join(tokens[:4]) if len(tokens) >= 4 else n2
    n4 = '.'.join(tokens) if len(tokens) >= 1 else ''
    return n1, n2, n3, n4

def fmt_data_nf(v):
    s = str(v).strip()
    if not s or set(s) == {'#'}:
        return ''
    dt = pd.to_datetime(s, format='%d/%m/%Y', errors='coerce')
    if pd.isna(dt):
        return ''
    return dt.strftime('%d/%m/%Y')

def fmt_data_pagamento(v):
    s = str(v).strip()
    if not s or s.upper() == 'NAO PAGO' or s in {'1800-01-01', '1900-01-01'}:
        return ''
    dt = pd.to_datetime(s, errors='coerce')
    if pd.isna(dt):
        return ''
    return dt.strftime('%d/%m/%Y')


In [3]:
def read_csv_with_fallback(path, sep=';', dtype=str, nrows=None):
    encodings = ['utf-8', 'utf-8-sig', 'cp1252', 'latin1']
    ultimo_erro = None
    for enc in encodings:
        try:
            df = pd.read_csv(path, sep=sep, dtype=dtype, encoding=enc, nrows=nrows, low_memory=False)
            return df, enc
        except UnicodeDecodeError as e:
            ultimo_erro = e
    raise UnicodeDecodeError(
        getattr(ultimo_erro, 'encoding', 'unknown'),
        getattr(ultimo_erro, 'object', b''),
        getattr(ultimo_erro, 'start', 0),
        getattr(ultimo_erro, 'end', 1),
        f'Nao foi possivel ler com encodings {encodings}: {ultimo_erro}'
    )

base, enc_base = read_csv_with_fallback(BASE_PATH)
modelo_header, enc_modelo = read_csv_with_fallback(MODELO_PATH, nrows=0)
modelo_cols = modelo_header.columns.tolist()
# Remove espacos nos cabecalhos (o modelo vem com " valor_conta ", " Valor Oficial ", etc.)
modelo_cols = [c.strip() if isinstance(c, str) else c for c in modelo_cols]
# Renomeia cabecalho legado do modelo para o nome correto
modelo_cols = ['Segmento' if c == 'sdssds' else c for c in modelo_cols]

print(f'Encoding base.csv detectado: {enc_base}')
print(f'Encoding modelo detectado: {enc_modelo}')

mes, ano = MES_REFERENCIA.split('/')
mes_int = int(mes)
ano_int = int(ano)

if CAMPO_DATA_FILTRO == 'lancamento':
    serie_data = base[CAMPO_DATA_FILTRO].fillna('').astype(str).str.strip()

    # 1) tenta parse de data dd/mm/aaaa
    dt = pd.to_datetime(serie_data, dayfirst=True, errors='coerce')
    filtro = (dt.dt.month == mes_int) & (dt.dt.year == ano_int)

    # 2) fallback textual para casos fora do padrao
    if not filtro.any():
        m2 = str(mes_int).zfill(2)
        padrao1 = rf'(^|\D){m2}/{ano_int}(\D|$)'
        padrao2 = rf'(^|\D){mes_int}/{ano_int}(\D|$)'
        filtro = serie_data.str.contains(padrao1, regex=True, na=False) | serie_data.str.contains(padrao2, regex=True, na=False)

elif CAMPO_DATA_FILTRO == 'ite_pagrec_vencimento':
    serie_data = base[CAMPO_DATA_FILTRO].fillna('').astype(str).str.strip()

    # 1) tenta parse de data aaaa-mm-dd
    dt = pd.to_datetime(serie_data, errors='coerce')
    filtro = (dt.dt.month == mes_int) & (dt.dt.year == ano_int)

    # 2) fallback textual
    if not filtro.any():
        m2 = str(mes_int).zfill(2)
        filtro = serie_data.str.startswith(f'{ano_int}-{m2}')
else:
    raise ValueError("CAMPO_DATA_FILTRO deve ser 'lancamento' ou 'ite_pagrec_vencimento'.")

df = base.loc[filtro].copy().reset_index(drop=True)
if df.empty:
    raise ValueError(
        f"Nenhum registro encontrado para {MES_REFERENCIA} usando {CAMPO_DATA_FILTRO}. "
        "Tente CAMPO_DATA_FILTRO='ite_pagrec_vencimento' ou confira o mes/ano."
    )

df['valor_bruto_num'] = df['valor_bruto'].apply(to_float_br)
df['valor_plano_num'] = df['valor_plano'].apply(to_float_br)
df['valor_plano_num'] = -df['valor_plano_num'].abs()
df['iterea_valpago_num'] = df['iterea_valpago'].apply(to_float_br)

out = pd.DataFrame(index=df.index)
out['id'] = (df.index + 1).astype(str).str.zfill(6)

lvl = df['codcen'].apply(levels_from_codcen)
out['n1_cod_centro_custo'] = lvl.apply(lambda x: x[0])
out['n2_cod_centro_custo'] = lvl.apply(lambda x: x[1])
out['n3_cod_centro_custo'] = lvl.apply(lambda x: x[2])
out['n4_cod_centro_custo'] = lvl.apply(lambda x: x[3])

desc_split = df['descen'].apply(split_descen)
out['n1_centro_custo'] = desc_split.apply(lambda x: x[1])
out['n2_centro_custo'] = desc_split.apply(lambda x: x[2])
out['n3_centro_custo'] = desc_split.apply(lambda x: x[3])
out['n4_centro_custo'] = desc_split.apply(lambda x: x[4])

out['Segmento'] = out['n1_centro_custo']
out['n1_CC'] = (out['n1_cod_centro_custo'].fillna('') + ' ' + out['n1_centro_custo'].fillna('')).str.strip()
out['n2_CC'] = (out['n2_cod_centro_custo'].fillna('') + ' ' + out['n2_centro_custo'].fillna('')).str.strip()
out['n3_CC'] = (out['n3_cod_centro_custo'].fillna('') + ' ' + out['n3_centro_custo'].fillna('')).str.strip()
out['n4_CC'] = (out['n4_cod_centro_custo'].fillna('') + ' ' + out['n4_centro_custo'].fillna('')).str.strip()

out['cod_conta'] = df['codcdc'].fillna('')
out['conta'] = df['descdc'].fillna('')
out['cod_conta-descr'] = (out['cod_conta'] + ' ' + out['conta']).str.strip()
out['filial'] = df['filial'].fillna('')
out['titulo'] = df['documento'].fillna('')
out['valor_nf'] = df['valor_bruto_num'].apply(to_br_number)
out['valor_pago'] = df['iterea_valpago_num'].apply(to_br_number)
out['valor_conta'] = df['valor_plano_num'].apply(to_br_number)
out['observacao'] = df['observacao'].fillna('')
out['data_nf'] = df['lancamento'].apply(fmt_data_nf)
out['data_pagamento'] = df['iterea_pagamento'].apply(fmt_data_pagamento)
out['cod_credor_forn_cli_func'] = df['codigo_pessoa'].fillna('')
out['credor_forn_cli_func'] = df['nome'].fillna('')

out['Origem'] = np.where(
    out['cod_conta'].str.startswith(('4.', '5.')),
    'Entrada (Origem)',
    'Saida (Aplicacoes)'
)
out['Sistema'] = 'SAGI'
out['Dados auxiliares'] = df['nota'].fillna('')
out['Valor Oficial'] = df['valor_bruto_num'].apply(to_br_currency)
out['DE-PARA1'] = ''
out['DE-PARA2'] = ''
out['CUSTEIO VARIAVEL'] = ''

# Garante o mesmo nome da coluna acentuada do modelo
if 'CUSTEIO VARIÁVEL' in modelo_cols and 'CUSTEIO VARIAVEL' in out.columns:
    out = out.rename(columns={'CUSTEIO VARIAVEL': 'CUSTEIO VARIÁVEL'})

# Reordena para o padrao do fechamento
for c in modelo_cols:
    if c not in out.columns:
        out[c] = ''
out = out[modelo_cols].fillna('')

nome_saida = f'FECHAMENTO_ODBC_{ano}_{mes}.xlsx'
saida = OUT_DIR / nome_saida

# Exporta e formata cabecalho (linha 1) em negrito
with pd.ExcelWriter(saida, engine='openpyxl') as writer:
    out.to_excel(writer, index=False, sheet_name='Fechamento')
    ws = writer.book['Fechamento']
    from openpyxl.styles import Font
    for cell in ws[1]:
        cell.font = Font(bold=True)

print(f'Registros convertidos: {len(out):,}'.replace(',', '.'))
print(f'Arquivo salvo em: {saida.resolve()}')
display(out.head(5))


Encoding base.csv detectado: latin1
Encoding modelo detectado: cp1252


C:\Users\julio.santana\AppData\Local\Temp\ipykernel_21568\527855638.py:37: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  dt = pd.to_datetime(serie_data, dayfirst=True, errors='coerce')


Registros convertidos: 6.081
Arquivo salvo em: C:\Users\julio.santana\Documents\Projects\Cofre_Trabalho\02-Referencias\FECHAMENTO_ODBC_2026_04.xlsx


,id,Segmento,n1_cod_centro_custo,n1_centro_custo,n1_CC,n2_cod_centro_custo,n2_centro_custo,n2_CC,n3_cod_centro_custo,n3_centro_custo,...,Unnamed: 43,Unnamed: 44,Unnamed: 45,Unnamed: 46,Unnamed: 47,Unnamed: 48,Unnamed: 49,Unnamed: 50,Unnamed: 51,Unnamed: 52
0,000001,SELETIVA,2.2,SELETIVA,2.2 SELETIVA,2.2.7,CAMPO GRANDE,2.2.7 CAMPO GRANDE,2.2.7.2,COMERCIAL,...,,,,,,,,,,
1,000002,SELETIVA,1.2,SELETIVA,1.2 SELETIVA,1.2.5,PRESIDENTE PRUDENTE,1.2.5 PRESIDENTE PRUDENTE,1.2.5.2,COMERCIAL,...,,,,,,,,,,
2,000003,SELETIVA,1.2,SELETIVA,1.2 SELETIVA,1.2.5,PRESIDENTE PRUDENTE,1.2.5 PRESIDENTE PRUDENTE,1.2.5.2,COMERCIAL,...,,,,,,,,,,
3,000004,SELETIVA,2.2,SELETIVA,2.2 SELETIVA,2.2.2,DOURADOS,2.2.2 DOURADOS,2.2.2.2,COMERCIAL,...,,,,,,,,,,
4,000005,SELETIVA,2.2,SELETIVA,2.2 SELETIVA,2.2.5,PRESIDENTE PRUDENTE,2.2.5 PRESIDENTE PRUDENTE,2.2.5.2,COMERCIAL,...,,,,,,,,,,


In [4]:
# Checagem rapida de aderencia ao modelo
faltantes = [c for c in modelo_cols if c not in out.columns]
extras = [c for c in out.columns if c not in modelo_cols]
print('Colunas faltantes:', faltantes)
print('Colunas extras:', extras)
print('Quantidade de colunas no modelo:', len(modelo_cols))
print('Quantidade de colunas na saida:', len(out.columns))


Colunas faltantes: []
Colunas extras: []
Quantidade de colunas no modelo: 53
Quantidade de colunas na saida: 53
